In [ ]:
# ------------------------------------------------------------------------------
# Standard Library Imports
# ------------------------------------------------------------------------------
import sys
import gc
import os
import re
from pathlib import Path
from collections import Counter

# ------------------------------------------------------------------------------
# Append Subfolder to Path for Local Modules
# ------------------------------------------------------------------------------
sys.path.append("../../src/")  # Adjust the path if your modules are in a different folder

# ------------------------------------------------------------------------------
# Local Module Imports
# ------------------------------------------------------------------------------
import functions  # Local module (e.g., functions.py inside src/)
from functions import *  # Import all functions from file.py

# ------------------------------------------------------------------------------
# Third-Party Library Imports
# ------------------------------------------------------------------------------
import torch
import torch.nn as nn
from torch import topk
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA


# ------------------------------------------------------------------------------
# adabmDCA Package Imports
# ------------------------------------------------------------------------------
from adabmDCA.fasta import get_tokens, write_fasta, import_from_fasta, encode_sequence, compute_weights
from adabmDCA.utils import get_device, get_dtype
from adabmDCA.stats import get_freq_single_point, get_freq_two_points, get_correlation_two_points
from adabmDCA.dataset import DatasetDCA
from adabmDCA.functional import one_hot
from adabmDCA.statmech import compute_energy
from adabmDCA.io import load_params

# ------------------------------------------------------------------------------
# arDCA Package Imports
# ------------------------------------------------------------------------------
import arDCA_paths
from arDCA_paths import arDCA_paths
from arDCA_paths.models import energy_third, energy_third_conditioned_first

# ------------------------------------------------------------------------------
# Transformer Package Imports
# ------------------------------------------------------------------------------
# from ProteinsDataset import ProteinTranslationDataset #*
# from ProteinTransformer import *
# from src.utils import *
import json
import pandas as pd


# ------------------------------------------------------------------------------
# Device and Data Type Settings
# ------------------------------------------------------------------------------
device = get_device("cpu")
device2 = get_device("cuda")
dtype = get_dtype("float32")

plt.rcParams.update({
    "text.usetex": True,
})


## Prediction Functions

Define three sequence prediction strategies:
- `predict_naive`: direct-path heuristic — fills the third segment using a per-position majority vote between first and second.
- `predict`: autoregressive arDCA sampling (stochastic decoding).
- `predict` with `ML=True`: arDCA greedy (maximum-likelihood) decoding.

In [ ]:
def predict_naive(data):
  
    X = data.clone()  # clone to avoid modifying input
    B, L, q = X.shape  # B,L,q: batch, length, alphabet size
    l = L // 3  # assume L is a multiple of 3

    first  = X[:, :l, :]      # shape (B, l, q)
    second = X[:, l:2*l, :]   # shape (B, l, q)

    third = first.clone()     # (B, l, q)
    mask_equal = (first == second).all(dim=2)      # (B, l), True if identical
    rand = torch.rand(B, l, device=X.device)       # (B, l)
    mask_replace = (~mask_equal) & (rand >= 0.5)    # (B, l), True => replace with second
    third[mask_replace] = second[mask_replace]  # copy from second where mask true

    X[:, 2*l:3*l, :] = third  # place predicted third into last third

    return X

def predict(data, model, ML=False, beta=1, device="cpu"):  # Fill last third autoregressively with arDCA
    """Predict using arDCA by sequentially filling the last third of the sequence."""
    model.to(device)  # Move model to target device
    X = data.clone().to(device)  # Clone input tensor on device
    L = X.shape[1]  # Total sequence length
    l = model.L // 3  # Third length from model
    for i in range(L - l, L):  # Iterate positions in last third
        prob = model.forward(X[:, :i, :], beta=beta)  # Conditional probs for position i
        sample = prob.argmax(dim=1) if ML else torch.multinomial(prob, num_samples=1).squeeze()  # Greedy or sampled index
        X[:, i] = nn.functional.one_hot(sample, model.q).to(dtype=model.h.dtype)  # Write one-hot token at i
    
    model.to("cpu")  # Restore model to CPU
    return X.to("cpu")  # Return predicted sequences on CPU


def predict_in_direct(data, model, ML=False, beta=1, device="cpu"):
    """Predict using arDCA by sequentially filling the last third of the sequence."""
    model.to(device)
    X = data.clone().to(device)
    L = X.shape[1]
    l = model.L // 3

    first = X[:, :l, :].argmax(dim=-1)      # dimensione (B, l)
    second = X[:, l:2*l, :].argmax(dim=-1)   # dimensione (B, l)
    mask_direct = (first != second)  # (B, l), True se diversi

    for i in range(L - l, L):
        prob = model.forward(X[:, :i, :], beta=beta)
        sample = prob.argmax(dim=1) if ML else torch.multinomial(prob, num_samples=1).squeeze()
        # Applica il mascheramento: mantieni il token originale se non è un buco
        pos_in_third = i - 2*l
        sample = torch.where(mask_holes[:, pos_in_third], sample, X[:, pos_in_third, :].argmax(dim=-1))
        X[:, i] = nn.functional.one_hot(sample, model.q).to(dtype=model.h.dtype)
    
    model.to("cpu")
    return X.to("cpu")

def predict_in_holes(data, model, ML=False, beta=1, device="cpu"):
    """Predict using arDCA by sequentially filling the last third of the sequence."""
    model.to(device)
    X = data.clone().to(device)
    L = X.shape[1]
    l = model.L // 3

    first = X[:, :l, :].argmax(dim=-1)      # dimensione (B, l)
    second = X[:, l:2*l, :].argmax(dim=-1)   # dimensione (B, l)
    mask_holes = (first != second)  # (B, l), True se diversi

    for i in range(L - l, L):
        prob = model.forward(X[:, :i, :], beta=beta)
        sample = prob.argmax(dim=1) if ML else torch.multinomial(prob, num_samples=1).squeeze()
        # Applica il mascheramento: mantieni il token originale se non è un buco
        pos_in_third = i - 2*l
        sample = torch.where(mask_holes[:, pos_in_third], sample, X[:, pos_in_third, :].argmax(dim=-1))
        X[:, i] = nn.functional.one_hot(sample, model.q).to(dtype=model.h.dtype)
    
    model.to("cpu")
    return X.to("cpu")

def predict_restricted_tokens(data, model, ML=False, beta=1, device="cpu"):
    """Predict using arDCA by sequentially filling the last third of the sequence,
    sampling only from tokens present in corresponding positions of first and second thirds."""
    model.to(device)  # Move model to specified device
    X = data.clone().to(device)  # Clone and move data to device
    L = X.shape[1]  # Sequence length
    l = model.L // 3  # Length of one third
    B = X.shape[0]  # Batch size
    
    mean_prob_cumul = 0.0  # Initialize cumulative probability sum
    with torch.no_grad():  # Disable gradient computation
        for i in range(L - l, L):  # Loop over last third positions
            prob = model.forward(X[:, :i, :], beta=beta)  # Get model probabilities (B, q)
            pos_first = i - 2*l  # Position in first third
            pos_second = i - l  # Position in second third

            token_first = X[:, pos_first].argmax(dim=-1)   # Get token from first third (B,)
            token_second = X[:, pos_second].argmax(dim=-1)  # Get token from second third (B,)
            
            valid_mask = torch.zeros(B, model.q, device=X.device, dtype=torch.bool)  # Mask for valid tokens
            valid_mask[torch.arange(B), token_first]  = True  # Allow first token
            valid_mask[torch.arange(B), token_second] = True  # Allow second token
            
            masked_prob = torch.where(valid_mask, prob, torch.zeros_like(prob))  # Mask probabilities
            
            mean_prob_cumul += masked_prob.sum(dim=-1, keepdim=False).mean()  # Accumulate mean masked prob

            prob_sum = masked_prob.sum(dim=-1, keepdim=True)  # Sum of masked probs
            prob_sum = torch.where(prob_sum > 0, prob_sum, torch.ones_like(prob_sum))  # Avoid zero division
            masked_prob = masked_prob / prob_sum  # Normalize masked probs
            
            has_valid_tokens = (prob_sum.squeeze(-1) > 0)  # Check for valid tokens
            final_prob = torch.where(has_valid_tokens.unsqueeze(-1), masked_prob, prob)  # Use masked or original probs
            
            sample = final_prob.argmax(dim=1) if ML else torch.multinomial(final_prob, num_samples=1).squeeze(-1)  # Sample or argmax
            
            X[:, i] = nn.functional.one_hot(sample, model.q).to(dtype=model.h.dtype)  # Set predicted token
        
    mean_prob_cumul /= l  # Average over third length
    model.to("cpu")  # Move model back to CPU
    return X.to("cpu"), mean_prob_cumul  # Return predictions and cumulative prob

### Accuracy Metrics

Define two accuracy functions:
- `compute_accuracy`: overall fraction of correctly predicted positions in the third segment, plus a normalised version restricted to mutated positions.
- `fraction_direct`: fraction of positions where the naive direct-path prediction matches the true third segment.

In [ ]:
def compute_accuracy(data, prediction):
    """Compute accuracy of third prediction, overall and normalized by disagreeing positions"""
    X = data.clone()
    B, L, q = X.shape
    l = L // 3  # assume L is a multiple of 3
    third = X[:, 2*l:3*l, :].clone() # (B, l, q)  # True third segment
    third_pred = prediction[:, 2*l:3*l, :].clone()   # (B, l, q)  # Predicted third segment
    accuracy = (third.argmax(dim=-1) == third_pred.argmax(dim=-1)).float().sum().item() / B  # Overall accuracy

    first = X[:, :l, :]      # shape (B, l, q)
    second = X[:, l:2*l, :]   # shape (B, l, q)

    different = (first != second).all(dim=2)  # (B, l), True if different
    n_different = different.sum(dim=1) # (B,)  # Count disagreeing positions per sample
    mask_nonzero = n_different != 0  # Filter samples with disagreements

    accuracy_norm =  (third.argmax(dim=-1) == third_pred.argmax(dim=-1)).float().sum(dim=1) / n_different # (B,)  # Normalize by disagreeing positions
    accuracy_norm = accuracy_norm[mask_nonzero].mean().item()  # Average over samples with disagreements

    return accuracy, accuracy_norm


def fraction_direct(data):
    """Compute fraction of third positions that match first or second directly"""
  
    X = data.clone()
    B, L, q = X.shape
    l = L // 3  # assume L is a multiple of 3
    first  = X[:, :l, :]      # shape (B, l, q)
    second = X[:, l:2*l, :]   # shape (B, l, q)
    third = X[:, 2*l:3*l, :].clone() # (B, l, q)

    # Check if third matches first or second at each position
    mask_direct = ((first == third) | (second == third)).all(dim=2)      # (B, l), True se identici
    frac_direct = mask_direct.float().mean().item()  # Fraction of direct matches
    
    return frac_direct

### Conservation-Diversity Entropy (CDE)

Implement `compute_CDE_full_batch_fast`, which computes per-position CDE scores for a batch of sequences using the bmDCA model. Processed in chunks to manage GPU memory.

In [ ]:
def compute_CDE_full_batch_fast(sequences, model, device=device, dtype=dtype, chunk_size=150):  # Compute conservation diversity entropy for sequences using model
    with torch.no_grad():  # Disable gradient computation for efficiency
        batch_size, L, q = sequences.shape  # Unpack sequence dimensions
        CDE_full_batch = torch.zeros((batch_size, L), device=device, dtype=dtype)  # Initialize output tensor for CDE values
 
        # chunk_size = 250  # Process in chunks to manage memory
        for start in range(0, batch_size, chunk_size):  # Iterate over chunks
            end = min(start + chunk_size, batch_size)  # End index for chunk
            sequences_chunk = sequences[start:end]  # Extract current chunk
            chunk_batch_size = sequences_chunk.shape[0]  # Size of current chunk
            model_device = {}  # Move model to device
            model_device["bias"] = model["bias"].to(device)  # Bias to device
            model_device["coupling_matrix"] = model["coupling_matrix"].to(device)  # Coupling matrix to device
            sequence_indices = sequences_chunk.argmax(-1)  # Get amino acid indices
            base = sequence_indices.unsqueeze(1).expand(-1, L * q, -1).reshape(chunk_batch_size * L * q, L).clone().to(device)  # Base sequences for mutations
            positions = torch.arange(L, device=device).repeat_interleave(q).repeat(chunk_batch_size)  # Positions for mutations
            values = torch.arange(q, device=device, dtype=torch.int64).repeat(L).repeat(chunk_batch_size)  # Amino acid values for mutations
            base[torch.arange(chunk_batch_size * L * q, device=device), positions] = values  # Set mutations in base
            dms = base  # Deep mutational scan sequences
            dms_oh = one_hot(dms, num_classes=q).to(dtype)  # One-hot encode DMS sequences
            energy_dms = compute_energy(dms_oh, model_device)  # Compute energies for DMS
            energy_dms = energy_dms.view(chunk_batch_size, L, q)  # Reshape to (batch, L, q)
            energy_dms -= energy_dms.min(dim=2, keepdim=True)[0]  # Normalize energies
            exp_energy_dms = torch.exp(-energy_dms)  # Exponentiate negative energies
            sums = exp_energy_dms.sum(dim=2, keepdim=True)  # Sum over amino acids
            p_i = exp_energy_dms / sums  # Probabilities
            CDE_chunk = -torch.sum(p_i * torch.log2(p_i), dim=2)  # Compute entropy (CDE)
            CDE_full_batch[start:end] = CDE_chunk.cpu()  # Store chunk result
            del base, positions, values, dms, dms_oh, energy_dms, exp_energy_dms, sums, p_i, model_device["bias"], model_device["coupling_matrix"], CDE_chunk  # Free memory
            
        # Clean memory  # Additional cleanup
        del sequences  # Delete input sequences
        gc.collect()  # Garbage collect
        torch.cuda.empty_cache()  # Clear CUDA cache
        
    return CDE_full_batch.cpu()  # Return CDE values on CPU

## Data Loading

Select the protein family and configure all input paths. Load the distance-stratified test FASTA files (one file per Hamming-distance bin) into `datasets`, and load the natural MSA and the reference bmDCA model.

In [ ]:
protein_family = "Chorismate Mutase" #"PF00076" # "Chorismate Mutase"
best = ""
patience = "patience5/"

if protein_family == "Chorismate Mutase":
    data_path = "../generated_data/CM"
    data_distance_path = "../generated_data/CM/reorganized_per_distance/new_filtering/non_filtered/" #"generated_data/CM/reorganized_per_distance/distance0T>0T2" #"generated_data/CM/reorganized_per_distance"
    cde_distance_path = "../generated_data/CM/reorganized_per_distance/new_filtering/non_filtered/cde_full_fast/"
    MSA_path = "../MSAs/CM_130530_MC.fasta"
    original_model_path = "../evolution_bmDCA_model/CM/params.dat"
    t = 0
    model_paths = f"../models_train_val/CM/{patience}"

    reg = "rJ1e-3_rH1e-5" #long_run_rJ1e-3_rH1e-5
    models_distance_paths = f"../models_train_val/CM/trained_per_distance/non_filtered/{patience}/" #"models/CM/per_distance/"
    LR_path = "../immagini_paper/CM/predict_time/LR_model_matched_positions_CDE0_CDET_full_30000_chains.joblib"
    output_path = f"../immagini_paper/CM/{patience}compare_distance/non_filtered/{best}_{reg}/"


# elif protein_family == "PF00076":
#     data_path = "generated_data/PF00076"
#     MSA_path = "MSAs/PF00076_mgap6.fasta"
#     original_model_path = "evolution_bmDCA_model/PF00076/params.dat"
#     t = 0
#     model_paths = "models/PF00076/"
#     # model_1cond_path = "models/PF00076/one_condition/"
#     output_path = "immagini_paper/PF00076/"
# elif protein_family == "PF00076_2nd":
#     data_path = "generated_data/PF00072"
#     MSA_path = "MSAs/PF00072.fasta"
#     original_model_path = "evolution_bmDCA_model/PF00072/params.dat"
#     t = 0
#     model_paths = "models/PF00072/"
#     # model_1cond_path = "models/PF00072/one_condition/"
#     output_path = "immagini_paper/PF00072/"
else:
    raise ValueError(f"Unknown protein family: {protein_family}")


In [ ]:
data_type = "ACDEFGHIKLMNPQRSTVWY-"
tokens = get_tokens(data_type)

timescales = ["10e1", "10e2", "10e3", "10e4", "10e5", "10e6"] 
d_list = list([ 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70])

# n_seqs = 1500

data_distance_files = [
    f"{data_distance_path}/distance_5_test.fasta",
    f"{data_distance_path}/distance_10_test.fasta",
    f"{data_distance_path}/distance_15_test.fasta",
    f"{data_distance_path}/distance_20_test.fasta",
    f"{data_distance_path}/distance_25_test.fasta",
    f"{data_distance_path}/distance_30_test.fasta",
    f"{data_distance_path}/distance_35_test.fasta",
    f"{data_distance_path}/distance_40_test.fasta",
    f"{data_distance_path}/distance_45_test.fasta",
    f"{data_distance_path}/distance_50_test.fasta",
    f"{data_distance_path}/distance_55_test.fasta",
    f"{data_distance_path}/distance_60_test.fasta",
    f"{data_distance_path}/distance_65_test.fasta",
    f"{data_distance_path}/distance_70_test.fasta",
    ]

datasets = {}
for d, path in zip(d_list, data_distance_files):
    ds = DatasetDCA(path_data=path, alphabet=data_type, device=device, dtype=dtype, no_reweighting=True)
    datasets[d] = ds.data
    del ds
    gc.collect()

print(f"Loaded {len(datasets)} datasets with {datasets[list(datasets.keys())[0]].shape[0]} sequences each")

In [ ]:
q = 21
print("q =",q)

headers, msa_enc_nat = import_from_fasta(MSA_path , tokens=tokens, filter_sequences=True)
msa_oh_nat = one_hot(torch.tensor(msa_enc_nat, device=device, dtype=torch.int32), num_classes=q).to(dtype)
print(datasets[5].shape)
M, L3 = datasets[5].shape
L = L3 // 3

# Load bmDCA model
print("Loading bmDCA model from:", original_model_path)
original_tokens = get_tokens("ACDEFGHIKLMNPQRSTVWY-")
original_model = load_params(original_model_path, tokens=original_tokens, device=device, dtype=dtype)

### arDCA Models Trained on Timescale Data

Load one trained arDCA model per evolutionary timescale. These models are used by the LR-guided predictor to reconstruct the third segment after the LR assigns a timescale label.

In [ ]:
models = {}
for t in timescales:
    path_params = model_paths
    params = torch.load(model_paths + f"{t}_{reg}/params{best}.pth", map_location='cpu')
    params['entropic_order'] = torch.arange(0, L3, dtype=torch.long, device='cpu')
    params['inverse_entropic_order'] = torch.arange(0, L3, dtype=torch.long, device='cpu')
    model = arDCA_paths(L=L3, q=q).to(device='cpu', dtype=dtype)
    model.load_state_dict(params, strict=False)
    models[t] = model

print(f"Loaded {len(models)} arDCA models")

### arDCA Models Trained on Distance-Stratified Data

Load one trained arDCA model per Hamming-distance bin. These models are trained directly on sequences grouped by their start-to-end distance, providing the distance-conditioned baseline.

In [ ]:
models_dist = {}
for d in d_list:
    path_params = model_paths
    params = torch.load(models_distance_paths + f"/model_d_upto_{d}_{reg}/params{best}.pth", map_location='cpu')
    params['entropic_order'] = torch.arange(0, L3, dtype=torch.long, device='cpu')
    params['inverse_entropic_order'] = torch.arange(0, L3, dtype=torch.long, device='cpu')
    model = arDCA_paths(L=L3, q=q).to(device='cpu', dtype=dtype)
    model.load_state_dict(params, strict=False)
    models_dist[d] = model

print(f"Loaded {len(models_dist)} arDCA models")

### Logistic Regression (LR) Model

Load the pretrained LR classifier from disk. The LR predicts which evolutionary timescale a given sequence belongs to, based on the mismatch vector between first and second segments combined with the CDE features.

In [ ]:
# load LR model example
import joblib

model_data = joblib.load(LR_path)
lr_model = model_data['model']
lr_scaler = model_data['scaler']
lr_features = model_data['features']  


### Output Directory

Create the output directory for saving figures and tables.

In [ ]:
Temperature = 1.0
beta = 1 / Temperature


os.makedirs(output_path, exist_ok=True)

## LR-Guided Prediction

Use the LR classifier to assign each sequence to a timescale, then reconstruct the third segment with the arDCA model corresponding to that timescale. This implements the full LR-guided prediction pipeline.

In [ ]:
def compute_distance_vector(sample):
    """Compute binary mismatch vector between first and second thirds"""
    l = sample.shape[1] // 3
    a, b, c = sample.split(l, dim=1)
    # Returns binary vector: 1 if mismatch, 0 if match
    # Since sample is already indices (B, L), not one-hot
    mismatch_vector = (a != b).float()  # (B, l)
    return mismatch_vector

print("Computing data distance vectors...")  # Notify start of distance computation
distance_vector = {  # Dict of mismatch vectors per key
    key: compute_distance_vector(data)  # Binary vector for each sample
    for key, data in datasets.items()}  # Filter datasets by timescales

In [ ]:
def load_distance_cde_dict(path, datasets=datasets, d_list=d_list, dtype=dtype):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"CDE file not found: {path}")
    loaded = np.load(path, allow_pickle=True).item()
    missing = [key for key in d_list if key not in loaded]
    if missing:
        raise KeyError(f"Missing distances in {path}: {missing}")

    cde = {}
    for key in d_list:
        values = loaded[key]
        n_values = datasets[key].shape[0]
        if values.shape[0] != n_values:
            raise ValueError(
                f"CDE file {path} for distance {key} has {values.shape[0]} values, "
                f"but dataset has {n_values} sequences."
            )
        cde[key] = torch.as_tensor(values, dtype=dtype)
    return cde


cde_distance_path = Path(cde_distance_path)
CDE_full_0 = load_distance_cde_dict(cde_distance_path / "CDE_0.npy")
CDE_mean_0 = load_distance_cde_dict(cde_distance_path / "CDE_0_mean.npy")
CDE_full_T = load_distance_cde_dict(cde_distance_path / "CDE_T.npy")
CDE_mean_T = load_distance_cde_dict(cde_distance_path / "CDE_T_mean.npy")


In [ ]:
print(f"Loaded LR model with {len(lr_features)} features")

# Create timescale mapping (same as in training)
def parse_timescale(ts):
    m = re.match(r'(\d+)e(\d+)', ts)
    base, exp = int(m.group(1)), int(m.group(2))
    return  (base ** exp)

# Assuming the model was trained on these timescales
# training_timescales = ["1x10e1", "1x10e2", "1x10e3", "1x10e4", "1x10e5"]
sorted_timescales = sorted(timescales, key=parse_timescale)
timescale_to_index = {ts: i for i, ts in enumerate(sorted_timescales)}
index_to_timescale = {i: ts for ts, i in timescale_to_index.items()}

print(f"Class to timescale mapping: {index_to_timescale}")

# Prepare predictions dictionary
predicted_LR = {}
predicted_LR_timescales = {}

for key in d_list:
    print(f"Processing LR prediction for distance: {key}")
    
    n_samples = datasets[key].shape[0]
    
    # Pre-allocate feature array (96 + 96 + 96 = 288 features)
    n_features = 96 + 96 + 96
    X_pred = np.zeros((n_samples, n_features), dtype=np.float32)
    
    # Fill features
    # Distance vector (96 features)
    X_pred[:, 0:96] = distance_vector[key].cpu().numpy()
    
    # CDE_0 (96 features)
    for i in range(n_samples):
        X_pred[i, 96:192] = CDE_full_0[key][i].cpu().numpy().flatten()
    
    # CDE_T (96 features)
    for i in range(n_samples):
        X_pred[i, 192:288] = CDE_full_T[key][i].cpu().numpy().flatten()
    
    # Create DataFrame with correct column names
    dist_cols = [f'dist_{j}' for j in range(96)]
    cde_0_cols = [f'CDE_0_{j}' for j in range(96)]
    cde_t_cols = [f'CDE_T_{j}' for j in range(96)]
    feature_names = dist_cols + cde_0_cols + cde_t_cols
    
    df_pred = pd.DataFrame(X_pred, columns=feature_names)
    
    # Scale features
    X_pred_scaled = lr_scaler.transform(df_pred[lr_features].values)
    
    # Make predictions (returns timescale class indices)
    predictions = lr_model.predict(X_pred_scaled)
    
    # Store predictions (both indices and timescale strings)
    predicted_LR[key] = predictions
    predicted_LR_timescales[key] = np.array([index_to_timescale[idx] for idx in predictions])
    
    print(f"Completed LR prediction for distance {key}: shape {predictions.shape}")
    print(f"Predicted timescales distribution: {np.unique(predicted_LR_timescales[key], return_counts=True)}")
    
    # Clean up
    del X_pred, df_pred, X_pred_scaled
    gc.collect()

print("LR predictions completed for all distances:", predicted_LR.keys())
print("\nTimescale mapping used:")
for idx, ts in index_to_timescale.items():
    print(f"  Class {idx} -> {ts}")

In [ ]:
# Group sequences by predicted timescale and predict with corresponding arDCA model
print("Predicting with LR-guided model selection...")

predicted_LR_guided = {}  # Store final predictions per distance
# predicted_LR_guidedML = {}  # Store final ML predictions per distance

for key in d_list:
    print(f"\nProcessing distance {key}...")
    
    # Get data for this distance
    data = datasets[key]
    n_samples = data.shape[0]
    
    # Get LR predictions for this distance (timescale strings)
    lr_preds = predicted_LR_timescales[key]
    
    # Initialize output tensor (clone original data structure)
    predictions =  one_hot(data.clone(), num_classes=q).to(dtype)
    # predictionsML = data.clone()
    
    # Group sequences by predicted timescale
    for ts in sorted_timescales:
        # Find indices where LR predicted this timescale
        mask = (lr_preds == ts)
        n_ts_samples = mask.sum()
        
        if n_ts_samples == 0:
            print(f"  No samples predicted for timescale {ts}")
            continue
            
        print(f"  Predicting {n_ts_samples} samples with model {ts}")
        # print(data.shape)

        # Extract sequences predicted for this timescale
        data_ts = data[mask] #one_hot(data[mask], num_classes=q).to(dtype)
        # Get the corresponding arDCA model
        model_ts = models[ts]
        print(data_ts.shape)
        
        # Predict with arDCA (sampling)
        data_ts = one_hot(data_ts, num_classes=q).to(dtype)
        pred_ts = predict(
            data_ts,
            model_ts,
            ML=False,
            beta=beta,
            device=device2
        )
  
        predictions[mask] =  pred_ts #torch.argmax(pred_ts, dim=-1).to(dtype=data.dtype)
    
    # Store predictions for this distance
    predicted_LR_guided[key] = predictions
    
    print(f"Completed predictions for distance {key}")

print("\nLR-guided predictions completed for all distances!")

### Distance-Conditioned arDCA Prediction

Reconstruct the third segment using the arDCA model trained on data with the matching Hamming distance. This is the distance-stratified baseline.

In [ ]:
datasets_oh = {}
for key in d_list:
    datasets_oh[key] = one_hot(datasets[key], num_classes=q).to(dtype)

In [ ]:
print("Predicting with arDCA...")
predicted_dist = {
    key: predict(
                data[:, models_dist[key].entropic_order.cpu().long().numpy(), :],
                models_dist[key],
                beta=beta,
                device=device2
            )[:, models_dist[key].inverse_entropic_order.cpu().long().numpy(), :]
    for key, data in datasets_oh.items()}


### Distance Metrics

Compute pairwise Hamming distances between the three segments `(S_start, S_mid, S_end)` for both the ground-truth data and each prediction method. Concatenate across all distance bins for population-level analysis.

In [ ]:
predicted_LR_guided_oh = {}
for key in d_list:
    predicted_LR_guided_oh[key] = predicted_LR_guided[key] #one_hot(predicted_LR_guided[key], num_classes=q).to(dtype)

In [ ]:
print("Computing data distances...")
distance = {
    key: torch.stack([ torch.tensor([
            L - compute_seqID(data[i][0:L],   data[i][2*L:]),
            L - compute_seqID(data[i][L:2*L], data[i][2*L:]),
            L - compute_seqID(data[i][0:L],   data[i][L:2*L]) ])
        for i in range(data.shape[0]) ])
    for key, data in datasets_oh.items()}

print("Computing model distances...")
# Compute distances for specific dataset keys using dictionary comprehension
distance_model = {
    key: torch.stack([
        torch.tensor([
            L - compute_seqID(data[i][0:L],   data[i][2*L:]),
            L - compute_seqID(data[i][L:2*L], data[i][2*L:]),
            # L - compute_seqID(data[i][0:L],   data[i][L:2*L])
        ])
        for i in range(data.shape[0])
    ])
    for key, data in predicted_LR_guided_oh.items()}


distance_model_dist = {
    key: torch.stack([
        torch.tensor([
            L - compute_seqID(data[i][0:L],   data[i][2*L:]),
            L - compute_seqID(data[i][L:2*L], data[i][2*L:]),
            # L - compute_seqID(data[i][0:L],   data[i][L:2*L])
        ])
        for i in range(data.shape[0])
    ])
    for key, data in predicted_dist.items()}

In [ ]:
all_CDE_0_mean = np.concatenate([CDE_mean_0[key] for key in d_list])
all_CDE_T_mean = np.concatenate([CDE_mean_T[key] for key in d_list])

d_0T2 = torch.cat([distance[key][:, 0] for key in d_list])
d_T2T = torch.cat([distance[key][:, 1] for key in d_list])
d_0T  = torch.cat([distance[key][:, 2] for key in d_list])

d_0T2_mdl = torch.cat([distance_model[key][:, 0] for key in d_list])
d_T2T_mdl = torch.cat([distance_model[key][:, 1] for key in d_list])

d_0T2_mdl_dist = torch.cat([distance_model_dist[key][:, 0] for key in d_list])
d_T2T_mdl_dist = torch.cat([distance_model_dist[key][:, 1] for key in d_list])



In [ ]:
plt.hist(all_CDE_0_mean, bins=30, edgecolor='black')
# add quantile lines 50%
plt.axvline(all_CDE_0_mean.mean().item(), color='red', linestyle='dashed', linewidth=1, label='Mean')
plt.axvline(np.median(all_CDE_0_mean), color='green', linestyle='dashed', linewidth=1, label='Median')
plt.xlabel("CDE_0 mean", fontsize=20)
plt.ylabel("Count", fontsize=20)
plt.title("Histogram of CDE_0 mean values", fontsize=20)
plt.legend(fontsize=16)
plt.grid(True, which="both", ls="--", alpha=0.2)
# plt.savefig(output_path + "/CDE_0_mean_histogram.png", dpi=300, bbox_inches='tight')
plt.show()

## CDE-Filtered Distance–Accuracy Plots

Plot reconstruction accuracy (and energy) as a function of Hamming distance, split by CDE regime (low vs. high). Each panel aggregates distance bins into 10-unit intervals.

In [ ]:
# Merge the two CDE-filtered plots (Low/High) into a single figure with 2 subplots
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

grouped_intervals = [
    (0, 10, '0-10'),
    (10, 20, '10-20'),
    (20, 30, '20-30'),
    (30, 40, '30-40'),
    (40, 50, '40-50'),
    (50, 60, '50-60'),
    (60, 70, '60-70')
]

palette = sns.color_palette("tab10")
colors_dict = {
    'data': palette[0],
    'arDCA LR': palette[4],
    'arDCA distance': palette[6]
}

thre_high = np.median(all_CDE_0_mean)
thre_low = np.median(all_CDE_0_mean)

cde_masks = [
    (all_CDE_0_mean <= thre_low, 'Low CDE'),
    (all_CDE_0_mean > thre_high, 'High CDE')
]

# ----------------------------
# Spacing controls
# ----------------------------
group_spacing_x = 0.90          # Spazio tra gruppi associati a valori diversi sull'asse x
bar_width_x = 0.16              # Larghezza orizzontale massima di ciascun mini-istogramma
intra_group_offset_x = 1.0      # Spazio relativo tra i 3 metodi dentro lo stesso gruppo
hist_width_scale = 0.75         # Scala di ampiezza dei profili istogramma rispetto a bar_width_x
bar_height_scale = 0.90         # Scala dell'altezza verticale delle barre nel profilo
subplot_wspace = 0.08           # Spazio tra subplot sinistro e destro

# ----------------------------
# Font-size controls
# ----------------------------
title_fs = 22
xlabel_fs = 28
ylabel_fs = 28
xtick_fs = 22
ytick_fs = 22
legend_fs = 24

# ----------------------------
# Grid controls
# ----------------------------
grid_alpha = 0.5
grid_linestyle = '--'

# Testi nei due subplot (in basso a destra)
corner_labels = [
    r"$\overline\mathrm{CDE}_\mathrm{start} <Q_{0.5}$",
    r"$\overline\mathrm{CDE}_\mathrm{start} > Q_{0.5}$"
]

fig, axs = plt.subplots(1, 2, figsize=(20, 7), sharey=True)
fig.subplots_adjust(wspace=subplot_wspace)

positions = np.arange(len(grouped_intervals)) * group_spacing_x
labels = ['data', 'arDCA LR', 'arDCA distance']

for ax_idx, (ax, (cde_mask, cde_title)) in enumerate(zip(axs, cde_masks)):
    all_values = []
    for d_min, d_max, _ in grouped_intervals:
        mask_distance = (d_0T >= d_min) & (d_0T < d_max)
        mask = mask_distance & cde_mask
        all_values.extend(d_0T2[mask].cpu().numpy())

    if len(all_values) > 0:
        bins = np.linspace(min(all_values), max(all_values), 30)
    else:
        bins = np.linspace(0, 100, 30)

    for i, (d_min, d_max, interval_label) in enumerate(grouped_intervals):
        mask_distance = (d_0T >= d_min) & (d_0T < d_max)
        mask = mask_distance & cde_mask

        data_list = [
            d_0T2[mask].cpu().numpy(),
            d_0T2_mdl[mask].cpu().numpy(),
            d_0T2_mdl_dist[mask].cpu().numpy()
        ]

        for j, (data_vals, method_label) in enumerate(zip(data_list, labels)):
            if len(data_vals) > 0:
                counts, bin_edges = np.histogram(data_vals, bins=bins, density=True)
                max_count = counts.max()
                counts_normalized = counts / max_count * bar_width_x * hist_width_scale if max_count > 0 else np.zeros_like(counts)
                offset = (j - 1) * bar_width_x * intra_group_offset_x
                bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

                for bc, cn in zip(bin_centers, counts_normalized):
                    if cn > 0:
                        ax.barh(
                            bc,
                            cn,
                            height=(bin_edges[1] - bin_edges[0]) * bar_height_scale,
                            left=positions[i] + offset - cn / 2,
                            color=colors_dict[method_label],
                            alpha=0.7,
                            edgecolor='black',
                            linewidth=0.5
                        )

                mean_val = data_vals.mean()
                ax.hlines(
                    mean_val,
                    positions[i] + offset - bar_width_x * 0.4,
                    positions[i] + offset + bar_width_x * 0.4,
                    colors='black',
                    linewidth=3,
                    zorder=10
                )

    ax.set_xticks(positions)
    ax.set_xticklabels([label for _, _, label in grouped_intervals], fontsize=xtick_fs)
    ax.set_xlabel(r'd($S_\mathrm{start}$, $S_\mathrm{end}$) intervals', fontsize=xlabel_fs, fontweight='normal')
    ax.tick_params(axis='y', labelsize=ytick_fs)
    ax.grid(True, which='both', ls=grid_linestyle, alpha=grid_alpha)
    ax.text(
        0.98,
        0.03,
        corner_labels[ax_idx],
        transform=ax.transAxes,
        ha='right',
        va='bottom',
        fontsize=title_fs
    )

axs[0].set_ylabel(r'd($S_\mathrm{start}$, $S_\mathrm{mid}$)', fontsize=ylabel_fs, fontweight='normal')

legend_handles = [
    Patch(facecolor=colors_dict['data'], edgecolor='black', alpha=0.7, label='data'),
    Patch(facecolor=colors_dict['arDCA LR'], edgecolor='black', alpha=0.7, label=r'MLR + arDCA (time)'),
    Patch(facecolor=colors_dict['arDCA distance'], edgecolor='black', alpha=0.7, label=r'arDCA (distance)'),
    # Line2D([0], [0], color='black', lw=3, label='mean')
]
axs[0].legend(handles=legend_handles, fontsize=legend_fs, loc='upper left')

plt.tight_layout()
plt.savefig(output_path + "/d_0T2_barplot_grouped_intervals_low_high_CDE_combined.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Unified version (no CDE filter): same histogram style as the accuracy plot
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch

# Intervalli reali associati ai file: distance_5 -> [0,5), ..., distance_70 -> [65,70)
d_edges = np.arange(0, 75, 5)
base_intervals = [
    (int(d_edges[i]), int(d_edges[i + 1]))
    for i in range(len(d_edges) - 1)
]

# Group base intervals pairwise: 0-10, 10-20, ...
grouped_intervals = []
i = 0
while i < len(base_intervals):
    start = base_intervals[i][0]
    if i + 1 < len(base_intervals):
        end = base_intervals[i + 1][1]
        i += 2
    else:
        end = base_intervals[i][1]
        i += 1
    grouped_intervals.append((start, end, f"{start}-{end}"))

palette = sns.color_palette("tab10")
colors_dict = {
    'data': palette[0],
    'arDCA LR': palette[4],
    'arDCA distance': palette[6]
}

# Spacing controls
plot_figsize = (14, 7)
group_spacing_x = 0.5
bar_width_x = 0.14
intra_group_offset_x = 1.0
hist_width_scale = 0.75
bar_height_scale = 0.90

# Font-size controls
xlabel_fs = 28
ylabel_fs = 28
xtick_fs = 22
ytick_fs = 22
legend_fs = 24

# Grid controls
grid_alpha = 0.5
grid_linestyle = '--'

fig, ax = plt.subplots(1, 1, figsize=plot_figsize)

positions = np.arange(len(grouped_intervals)) * group_spacing_x
labels = ['data', 'arDCA LR', 'arDCA distance']

# Bin comuni su tutti i valori (no filtro CDE)
all_values = np.concatenate([
    d_0T2.cpu().numpy(),
    d_0T2_mdl.cpu().numpy(),
    d_0T2_mdl_dist.cpu().numpy()
])
bins = np.linspace(all_values.min(), all_values.max(), 30)

for i, (d_min, d_max, interval_label) in enumerate(grouped_intervals):
    mask_distance = (d_0T >= d_min) & (d_0T < d_max)

    data_list = [
        d_0T2[mask_distance].cpu().numpy(),
        d_0T2_mdl[mask_distance].cpu().numpy(),
        d_0T2_mdl_dist[mask_distance].cpu().numpy()
    ]

    for j, (data_vals, method_label) in enumerate(zip(data_list, labels)):
        if len(data_vals) == 0:
            continue

        counts, bin_edges = np.histogram(data_vals, bins=bins, density=True)
        max_count = counts.max()
        counts_normalized = counts / max_count * bar_width_x * hist_width_scale if max_count > 0 else np.zeros_like(counts)
        offset = (j - 1) * bar_width_x * intra_group_offset_x
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        for bc, cn in zip(bin_centers, counts_normalized):
            if cn > 0:
                ax.barh(
                    bc,
                    cn,
                    height=(bin_edges[1] - bin_edges[0]) * bar_height_scale,
                    left=positions[i] + offset - cn / 2,
                    color=colors_dict[method_label],
                    alpha=0.7,
                    edgecolor='black',
                    linewidth=0.5
                )

        mean_val = data_vals.mean()
        ax.hlines(
            mean_val,
            positions[i] + offset - bar_width_x * 0.4,
            positions[i] + offset + bar_width_x * 0.4,
            colors='black',
            linewidth=2.5,
            zorder=10
        )

ax.set_xticks(positions)
ax.set_xticklabels([label for _, _, label in grouped_intervals], fontsize=xtick_fs)
ax.set_xlabel(r'd($S_\mathrm{start}$, $S_\mathrm{end}$) intervals', fontsize=xlabel_fs, fontweight='normal')
ax.set_ylabel(r'd($S_\mathrm{start}$, $S_\mathrm{mid}$)', fontsize=ylabel_fs, fontweight='normal')
ax.tick_params(axis='y', labelsize=ytick_fs)
ax.grid(True, which='both', ls=grid_linestyle, alpha=grid_alpha)

legend_handles = [
    Patch(facecolor=colors_dict['data'], edgecolor='black', alpha=0.7, label='data'),
    Patch(facecolor=colors_dict['arDCA LR'], edgecolor='black', alpha=0.7, label=r'MLR + arDCA(time) sample'),
    Patch(facecolor=colors_dict['arDCA distance'], edgecolor='black', alpha=0.7, label=r'arDCA(distance) sample'),
]
ax.legend(handles=legend_handles, fontsize=legend_fs, loc='upper left')

plt.tight_layout()
plt.savefig(output_path + "/d_0T2_barplot_2bins_merged_no_CDE_filter.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# d_list = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70]
# n_intervals = len(d_list) - 1

# fig, axs = plt.subplots(nrows=n_intervals, ncols=3, figsize=(13, 2 * n_intervals), sharex=False, sharey=True)
# bins = np.arange(0, max(d_0T2.max().item(), d_0T2_mdl.max().item(), d_0T2_mdl_dist.max().item()) + 1, 3)

# data_sources = [(d_0T2, 'Data'), (d_0T2_mdl, 'Model'), (d_0T2_mdl_dist, 'Model (dist)')]

# for i in range(n_intervals):
#     d_min, d_max = d_list[i], d_list[i+1]
#     mask_distance = (d_0T >= d_min) & (d_0T < d_max)
#     mask_low = mask_distance & (all_CDE_0_mean <= thre_low)
#     mask_high = mask_distance & (all_CDE_0_mean > thre_high)
    
#     for j, (data, title) in enumerate(data_sources):
#         ax = axs[i, j]
        
#         # Plot histograms
#         ax.hist(data[mask_low].cpu().numpy(), bins=bins, alpha=0.6, color='blue', edgecolor='black', label='Low CDE', density=True)
#         ax.hist(data[mask_high].cpu().numpy(), bins=bins, alpha=0.6, color='red', edgecolor='black', label='High CDE', density=True)

#         # Plot means
#         if mask_low.sum() > 0:
#             mean_low = data[mask_low].cpu().numpy().mean()
#             ax.axvline(mean_low, color='blue', linestyle='--', linewidth=2)
#         if mask_high.sum() > 0:
#             mean_high = data[mask_high].cpu().numpy().mean()
#             ax.axvline(mean_high, color='red', linestyle='--', linewidth=2)
        
#         # Shade the interval
#         ax.axvspan(d_min, d_max, color='gray', alpha=0.3)
        
#         # Set labels and titles
#         ax.set_ylabel(f'{d_min} - {d_max}', fontsize=20)
#         if i == 0:
#             ax.legend(fontsize=20)
#             ax.set_title(title, fontsize=20)
#         ax.grid(True)

# # Set common x-labels
# axs[-1, 0].set_xlabel(r'$d(0,T/2)$', fontsize=20)
# axs[-1, 1].set_xlabel(r'$d(0,T/2)$', fontsize=20)

# plt.suptitle("Distributions of $d(0,T/2)$ filtered by CDE(t=0)", fontsize=24, y=1.01)
# plt.subplots_adjust(hspace=0, wspace=0)
# plt.tight_layout()
# # plt.savefig(output_path + "/d_0T2_histograms_per_d_0T_and_CDE_separated.png", dpi=300, bbox_inches='tight')
# plt.show()

## Energy Analysis

Compute the bmDCA energy of the predicted third segment for each method (LR-guided, distance-conditioned) and for the ground-truth data. Normalise energies and concatenate across distance bins.

In [ ]:
energies_data       = {t: compute_energy(data[:, 2*L:, :],   original_model) for t, data in datasets_oh.items()}
energies_data_start = {t: compute_energy(data[:, 0:L, :],    original_model) for t, data in datasets_oh.items()}

energies_model = {t: compute_energy(data[:, 2*L:, :], original_model) for t, data in predicted_LR_guided_oh.items()}
energies_model_dist = {t: compute_energy(data[:, 2*L:, :], original_model) for t, data in predicted_dist.items()}
# # energies_ML    = {t: compute_energy(data[:, 2*L:, :], original_model) for t, data in predictedML.items()}

energies_data_norm  = {t: energies_data[t]  - energies_data_start[t] for t in datasets_oh.keys()}
energies_model_norm = {t: energies_model[t] - energies_data_start[t] for t in predicted_LR_guided_oh.keys()}
energies_model_dist_norm = {t: energies_model_dist[t] - energies_data_start[t] for t in predicted_dist.keys()}
# energies_ML_norm    = {t: energies_ML[t]    - energies_data_start[t] for t in datasets.keys()}

energies_nat = compute_energy(msa_oh_nat, original_model) 

In [ ]:
energies_all = torch.cat([energies_data[key] for key in datasets.keys()])
energies_all_mdl = torch.cat([energies_model[key] for key in datasets.keys()])
energies_all_mdl_dist = torch.cat([energies_model_dist[key] for key in datasets.keys()])

## Accuracy and Energy Analysis

Redefine a per-sample accuracy function, compute per-sequence accuracy for LR-guided and distance-conditioned predictions, and plot the accuracy distributions stratified by Hamming distance bin.

In [ ]:
def compute_accuracy(data, prediction):
    """Compute overall accuracy of third segment prediction"""
    X = data.clone()
    B, L, q = X.shape
    l = L // 3  # assume L is a multiple of 3
    third = X[:, 2*l:3*l, :].argmax(dim=-1) # (B, l)  # True third tokens
    third_pred = prediction[:, 2*l:3*l, :].argmax(dim=-1)   # (B, l)  # Predicted third tokens
    first, second = X[:, :l, :].argmax(dim=-1), X[:, l:2*l, :].argmax(dim=-1)  # (B, l)
    mutated = (first != second)   # (B, l), True if the position mutated between start and end
    accuracy = (third == third_pred).float().sum(dim=-1) # Overall number correct
    mean_accuracy = accuracy.sum().item() / B   # Mean accuracy per token
    # n_matches_12 = (first == second).float().sum(dim=-1)  # Number of matches between first and second
    mask_mutated = [mutated[i] for i in range(B)]   # (B, l), True se diversi  # Third differs from eithe

    accuracy_renorm = 0
    for i in range(B):
        n_mutated = mask_mutated[i].sum().item()
        if n_mutated > 0:
            correct = (third[i][mask_mutated[i]] == third_pred[i][mask_mutated[i]]).float().sum().item()
            accuracy_renorm += (correct / n_mutated )
    accuracy_renorm /= B

    return mean_accuracy / l , accuracy_renorm


def compute_accuracy_norm(data, prediction):
    """Compute normalized accuracies for different disagreement patterns"""
    X = data.clone()
    B, L, q = X.shape
    l = L // 3  # assume L is a multiple of 3
    third = X[:, 2*l:3*l, :].argmax(dim=-1) # (B, l)  # True third tokens
    third_pred = prediction[:, 2*l:3*l, :].argmax(dim=-1)   # (B, l)  # Predicted third tokens

    first = X[:, :l, :].argmax(dim=-1)      # dimensione (B, l)  # First third tokens
    second = X[:, l:2*l, :].argmax(dim=-1)   # dimensione (B, l)  # Second third tokens

    different_intersec = ((first != third) & (second != third) )  # (B, l), True se diversi  # Third differs from both
    different_union = ((first != third) | (second != third) )  # (B, l), True se diversi  # Third differs from either
    different_from_first = (first != third)  # (B, l), True se diversi  # Third differs from first
    different_from_second = (second != third)  # (B, l), True se diversi  # Third differs from second

    different_0T = ((first != second))  # (B, l), True se diversi  # First and second differ

    n_different_intersec = different_intersec.sum(dim=1) # (B,)  # Count intersection disagreements
    n_different_union = different_union.sum(dim=1) # (B,)  # Count union disagreements
    n_different_from_first = different_from_first.sum(dim=1) # (B,)  # Count first disagreements
    n_different_from_second = different_from_second.sum(dim=1) # (B,)  # Count second disagreements
    n_different_0T = different_0T.sum(dim=1) # (B,)  # Count first-second disagreements

    accuracy_intersec = torch.zeros(B)  # Per-sample accuracy for intersection
    accuracy_union = torch.zeros(B)  # Per-sample accuracy for union
    accuracy_from_first = torch.zeros(B)  # Per-sample accuracy for first diffs
    accuracy_from_second = torch.zeros(B)  # Per-sample accuracy for second diffs
    accuracy_0T = torch.zeros(B)  # Per-sample accuracy for first-second diffs

    # Extract masked tokens for each disagreement type
    masked_intersec = [t[d] for t, d in zip(third, different_intersec)]
    masked_intersec_pred = [t[d] for t, d in zip(third_pred, different_intersec)]

    masked_union = [t[d] for t, d in zip(third, different_union)]
    masked_union_pred = [t[d] for t, d in zip(third_pred, different_union)]

    masked_from_first = [t[d] for t, d in zip(third, different_from_first)]
    masked_from_first_pred = [t[d] for t, d in zip(third_pred, different_from_first)]

    masked_from_second = [t[d] for t, d in zip(third, different_from_second)]
    masked_from_second_pred = [t[d] for t, d in zip(third_pred, different_from_second)]

    masked_0T = [t[d] for t, d in zip(third, different_0T)]
    masked_0T_pred = [t[d] for t, d in zip(third_pred, different_0T)]

    # Compute per-sample accuracies
    for i in range(B):
        if n_different_intersec[i] > 0:  # Avoid division by zero
            correct = (masked_intersec[i] == masked_intersec_pred[i]).float().sum().item()
            accuracy_intersec[i] = correct / n_different_intersec[i].item()
        if n_different_union[i] > 0:
            correct = (masked_union[i] == masked_union_pred[i]).float().sum().item()
            accuracy_union[i] = correct / n_different_union[i].item()
        if n_different_from_first[i] > 0:
            correct = (masked_from_first[i] == masked_from_first_pred[i]).float().sum().item()
            accuracy_from_first[i] = correct / n_different_from_first[i].item()
        if n_different_from_second[i] > 0:
            correct = (masked_from_second[i] == masked_from_second_pred[i]).float().sum().item()
            accuracy_from_second[i] = correct / n_different_from_second[i].item()
        if n_different_0T[i] > 0:
            correct = (masked_0T[i] == masked_0T_pred[i]).float().sum().item()
            accuracy_0T[i] = correct / n_different_0T[i].item()

    # Average over samples with nonzero disagreements
    accuracy_intersec = accuracy_intersec[n_different_intersec > 0].mean().item()
    accuracy_union = accuracy_union[n_different_union > 0].mean().item()
    accuracy_from_first = accuracy_from_first[n_different_from_first > 0].mean().item()
    accuracy_from_second = accuracy_from_second[n_different_from_second > 0].mean().item()
    accuracy_0T = accuracy_0T[n_different_0T > 0].mean().item()

    return accuracy_intersec, accuracy_union, accuracy_from_first, accuracy_from_second, accuracy_0T



acc_mdl = { key: compute_accuracy(data, predicted_LR_guided_oh[key]) for key, data in datasets_oh.items()}
acc_mdl_dist   = { key: compute_accuracy(data, predicted_dist[key]) for key, data in datasets_oh.items()}

In [ ]:
# Accuracy plot (vertical histogram style like cell 42)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def per_sample_accuracy(data, prediction):
    X = data
    B, L, q = X.shape
    l = L // 3
    third = X[:, 2 * l:3 * l, :].argmax(dim=-1)
    third_pred = prediction[:, 2 * l:3 * l, :].argmax(dim=-1)
    return (third == third_pred).float().mean(dim=1)

acc_lr = torch.cat([
    per_sample_accuracy(datasets_oh[key], predicted_LR_guided_oh[key])
    for key in d_list
])
acc_dist = torch.cat([
    per_sample_accuracy(datasets_oh[key], predicted_dist[key])
    for key in d_list
])

d_edges = np.arange(0, 75, 5)
base_intervals = [
    (int(d_edges[i]), int(d_edges[i + 1]))
    for i in range(len(d_edges) - 1)
 ]
grouped_intervals = []
i = 0
while i < len(base_intervals):
    start = base_intervals[i][0]
    if i + 1 < len(base_intervals):
        end = base_intervals[i + 1][1]
        i += 2
    else:
        end = base_intervals[i][1]
        i += 1
    grouped_intervals.append((start, end, f"{start}-{end}"))

group_spacing_x = 0.5
bar_width_x = 0.14
intra_group_offset_x = 1.0
hist_width_scale = 0.75
bar_height_scale = 0.90

xlabel_fs = 30
ylabel_fs = 30
xtick_fs = 26
ytick_fs = 26
legend_fs = 24

grid_alpha = 0.5
grid_linestyle = "--"

colors_dict = {
    "MLR + arDCA(time)": "C4",
    "arDCA(distance)": "C6"
}

positions = np.arange(len(grouped_intervals)) * group_spacing_x
labels = ["MLR + arDCA(time)", "arDCA(distance)"]

all_values = np.concatenate([
    acc_lr.cpu().numpy(),
    acc_dist.cpu().numpy()
 ])
bins = np.linspace(all_values.min(), all_values.max(), 30)

fig, ax = plt.subplots(1, 1, figsize=(14, 7))
for i, (d_min, d_max, interval_label) in enumerate(grouped_intervals):
    mask_distance = (d_0T >= d_min) & (d_0T < d_max)
    data_list = [
        acc_lr[mask_distance].cpu().numpy(),
        acc_dist[mask_distance].cpu().numpy()
    ]
    for j, (data_vals, method_label) in enumerate(zip(data_list, labels)):
        if len(data_vals) == 0:
            continue
        counts, bin_edges = np.histogram(data_vals, bins=bins, density=True)
        max_count = counts.max()
        counts_normalized = counts / max_count * bar_width_x * hist_width_scale if max_count > 0 else np.zeros_like(counts)
        offset = (j - 0.5) * bar_width_x * intra_group_offset_x
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        for bc, cn in zip(bin_centers, counts_normalized):
            if cn > 0:
                ax.barh(
                    bc,
                    cn,
                    height=(bin_edges[1] - bin_edges[0]) * bar_height_scale,
                    left=positions[i] + offset - cn / 2,
                    color=colors_dict[method_label],
                    alpha=0.7,
                    edgecolor="black",
                    linewidth=0.5
                )
        mean_val = data_vals.mean()
        ax.hlines(
            mean_val,
            positions[i] + offset - bar_width_x * 0.4,
            positions[i] + offset + bar_width_x * 0.4,
            colors="black",
            linewidth=2.5,
            zorder=10
        )

ax.set_xticks(positions)
ax.set_xticklabels([label for _, _, label in grouped_intervals], fontsize=xtick_fs)
ax.set_xlabel(r"d($S_\mathrm{start}$, $S_\mathrm{end}$) intervals", fontsize=xlabel_fs, fontweight="normal")
ax.set_ylabel("Accuracy", fontsize=ylabel_fs, fontweight="normal")
ax.tick_params(axis="y", labelsize=ytick_fs)
ax.grid(True, which="both", ls=grid_linestyle, alpha=grid_alpha)

legend_handles = [
    Patch(facecolor=colors_dict["MLR + arDCA(time)"], edgecolor="black", alpha=0.7, label="MLR + arDCA(time) ML"),
    Patch(facecolor=colors_dict["arDCA(distance)"], edgecolor="black", alpha=0.7, label="arDCA(distance) ML"),
 ]
ax.legend(handles=legend_handles, fontsize=legend_fs, loc="upper right")

plt.tight_layout()
plt.savefig(output_path + "/accuracy_barplot_2bins_merged.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Energy and Distance Comparison Plots

Repeat the accuracy-style histogram plots with the energy of the third segment on the y-axis, and produce a combined figure showing both distance and energy side by side.

In [ ]:
# Unified version (no CDE filter): same style as the accuracy plot but y-axis shows energy
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch

# Intervalli reali associati ai file: distance_5 -> [0,5), ..., distance_70 -> [65,70)
d_edges = np.arange(0, 75, 5)
base_intervals = [
    (int(d_edges[i]), int(d_edges[i + 1]))
    for i in range(len(d_edges) - 1)
]

# Group base intervals pairwise: 0-10, 10-20, ...
grouped_intervals = []
i = 0
while i < len(base_intervals):
    start = base_intervals[i][0]
    if i + 1 < len(base_intervals):
        end = base_intervals[i + 1][1]
        i += 2
    else:
        end = base_intervals[i][1]
        i += 1
    grouped_intervals.append((start, end, f"{start}-{end}"))

palette = sns.color_palette("tab10")
colors_dict = {
    'data': palette[0],
    'arDCA LR': palette[4],
    'arDCA distance': palette[6]
}

# Spacing controls
plot_figsize = (14, 7)
group_spacing_x = 0.5
bar_width_x = 0.14
intra_group_offset_x = 1.0
hist_width_scale = 0.75
bar_height_scale = 0.90

# Font-size controls
xlabel_fs = 28
ylabel_fs = 28
xtick_fs = 22
ytick_fs = 22
legend_fs = 24

# Grid controls
grid_alpha = 0.5
grid_linestyle = '--'

fig, ax = plt.subplots(1, 1, figsize=plot_figsize)

positions = np.arange(len(grouped_intervals)) * group_spacing_x
labels = ['data', 'arDCA LR', 'arDCA distance']

# Bin comuni su tutte le energie (no filtro CDE)
all_values = np.concatenate([
    energies_all.cpu().numpy(),
    energies_all_mdl.cpu().numpy(),
    energies_all_mdl_dist.cpu().numpy()
])
bins = np.linspace(all_values.min(), all_values.max(), 30)

for i, (d_min, d_max, interval_label) in enumerate(grouped_intervals):
    mask_distance = (d_0T >= d_min) & (d_0T < d_max)

    data_list = [
        energies_all[mask_distance].cpu().numpy(),
        energies_all_mdl[mask_distance].cpu().numpy(),
        energies_all_mdl_dist[mask_distance].cpu().numpy()
    ]

    for j, (data_vals, method_label) in enumerate(zip(data_list, labels)):
        if len(data_vals) == 0:
            continue

        counts, bin_edges = np.histogram(data_vals, bins=bins, density=True)
        max_count = counts.max()
        counts_normalized = counts / max_count * bar_width_x * hist_width_scale if max_count > 0 else np.zeros_like(counts)
        offset = (j - 1) * bar_width_x * intra_group_offset_x
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

        for bc, cn in zip(bin_centers, counts_normalized):
            if cn > 0:
                ax.barh(
                    bc,
                    cn,
                    height=(bin_edges[1] - bin_edges[0]) * bar_height_scale,
                    left=positions[i] + offset - cn / 2,
                    color=colors_dict[method_label],
                    alpha=0.7,
                    edgecolor='black',
                    linewidth=0.5
                )

        mean_val = data_vals.mean()
        ax.hlines(
            mean_val,
            positions[i] + offset - bar_width_x * 0.4,
            positions[i] + offset + bar_width_x * 0.4,
            colors='black',
            linewidth=2.5,
            zorder=10
        )

ax.set_xticks(positions)
ax.set_xticklabels([label for _, _, label in grouped_intervals], fontsize=xtick_fs)
ax.set_xlabel(r'd($S_\mathrm{start}$, $S_\mathrm{end}$) intervals', fontsize=xlabel_fs, fontweight='normal')
ax.set_ylabel(r'$E$', fontsize=ylabel_fs, fontweight='normal')
ax.tick_params(axis='y', labelsize=ytick_fs)
ax.grid(True, which='both', ls=grid_linestyle, alpha=grid_alpha)

legend_handles = [
    Patch(facecolor=colors_dict['data'], edgecolor='black', alpha=0.7, label='data'),
    Patch(facecolor=colors_dict['arDCA LR'], edgecolor='black', alpha=0.7, label=r'MLR + arDCA(time) sample'),
    Patch(facecolor=colors_dict['arDCA distance'], edgecolor='black', alpha=0.7, label=r'arDCA(distance) sample'),
]
ax.legend(handles=legend_handles, fontsize=legend_fs, loc='upper left')

plt.tight_layout()
plt.savefig(output_path + "/energy_barplot_2bins_merged_no_CDE_filter.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Distance and energy side by side with the same histogram style as the accuracy plot
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch

# Intervalli reali associati ai file: distance_5 -> [0,5), ..., distance_70 -> [65,70)
d_edges = np.arange(0, 75, 5)
base_intervals = [
    (int(d_edges[i]), int(d_edges[i + 1]))
    for i in range(len(d_edges) - 1)
]

# Group base intervals pairwise: 0-10, 10-20, ...
grouped_intervals = []
i = 0
while i < len(base_intervals):
    start = base_intervals[i][0]
    if i + 1 < len(base_intervals):
        end = base_intervals[i + 1][1]
        i += 2
    else:
        end = base_intervals[i][1]
        i += 1
    grouped_intervals.append((start, end, f"{start}-{end}"))

palette = sns.color_palette("tab10")
colors_dict = {
    'data': palette[0],
    'arDCA LR': palette[4],
    'arDCA distance': palette[6]
}
labels = ['data', 'arDCA LR', 'arDCA distance']
legend_handles = [
    Patch(facecolor=colors_dict['data'], edgecolor='black', alpha=0.7, label='data'),
    Patch(facecolor=colors_dict['arDCA LR'], edgecolor='black', alpha=0.7, label=r'MLR + arDCA(time) sample'),
    Patch(facecolor=colors_dict['arDCA distance'], edgecolor='black', alpha=0.7, label=r'arDCA(distance) sample'),
]

# Stile coerente con il plot di accuracy
combined_figsize = (20, 7)
group_spacing_x = 0.5
bar_width_x = 0.14
intra_group_offset_x = 1.0
hist_width_scale = 0.75
bar_height_scale = 0.90
xlabel_fs = 28
ylabel_fs = 28
xtick_fs = 22
ytick_fs = 22
legend_fs = 24
grid_alpha = 0.5
grid_linestyle = '--'

positions = np.arange(len(grouped_intervals)) * group_spacing_x

plot_specs = [
    (
        r'd($S_\mathrm{start}$, $S_\mathrm{mid}$)',
        [d_0T2, d_0T2_mdl, d_0T2_mdl_dist],
        np.concatenate([d_0T2.cpu().numpy(), d_0T2_mdl.cpu().numpy(), d_0T2_mdl_dist.cpu().numpy()])
    ),
    (
        r'Energy',
        [energies_all, energies_all_mdl, energies_all_mdl_dist],
        np.concatenate([energies_all.cpu().numpy(), energies_all_mdl.cpu().numpy(), energies_all_mdl_dist.cpu().numpy()])
    )
]

fig, axs = plt.subplots(1, 2, figsize=combined_figsize)

for ax, (ylabel, value_sets, all_values) in zip(axs, plot_specs):
    bins = np.linspace(all_values.min(), all_values.max(), 30)

    for i, (d_min, d_max, interval_label) in enumerate(grouped_intervals):
        mask_distance = (d_0T >= d_min) & (d_0T < d_max)
        data_list = [values[mask_distance].cpu().numpy() for values in value_sets]

        for j, (data_vals, method_label) in enumerate(zip(data_list, labels)):
            if len(data_vals) == 0:
                continue

            counts, bin_edges = np.histogram(data_vals, bins=bins, density=True)
            max_count = counts.max()
            counts_normalized = counts / max_count * bar_width_x * hist_width_scale if max_count > 0 else np.zeros_like(counts)
            offset = (j - 1) * bar_width_x * intra_group_offset_x
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

            for bc, cn in zip(bin_centers, counts_normalized):
                if cn > 0:
                    ax.barh(
                        bc,
                        cn,
                        height=(bin_edges[1] - bin_edges[0]) * bar_height_scale,
                        left=positions[i] + offset - cn / 2,
                        color=colors_dict[method_label],
                        alpha=0.7,
                        edgecolor='black',
                        linewidth=0.5
                    )

            mean_val = data_vals.mean()
            ax.hlines(
                mean_val,
                positions[i] + offset - bar_width_x * 0.4,
                positions[i] + offset + bar_width_x * 0.4,
                colors='black',
                linewidth=2.5,
                zorder=10
            )

    ax.set_xticks(positions)
    ax.set_xticklabels([label for _, _, label in grouped_intervals], fontsize=xtick_fs)
    ax.set_xlabel(r'd($S_\mathrm{start}$, $S_\mathrm{end}$) intervals', fontsize=xlabel_fs, fontweight='normal')
    ax.set_ylabel(ylabel, fontsize=ylabel_fs, fontweight='normal')
    ax.tick_params(axis='y', labelsize=ytick_fs)
    ax.grid(True, which='both', ls=grid_linestyle, alpha=grid_alpha)

axs[0].legend(handles=legend_handles, fontsize=legend_fs, loc='upper left')

plt.tight_layout()
plt.savefig(output_path + "/distance_energy_barplot_2bins_merged_no_CDE_filter.pdf", dpi=300, bbox_inches='tight')
plt.show()
